# Dynamic Pricing Env

> Static dynamic pricing environment where a decision only affects the next period 

In [ ]:
#| default_exp envs.pricing.dynamic

In [1]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from abc import ABC, abstractmethod
from typing import Union, Tuple, Literal

from ddopai.utils import Parameter, MDPInfo
from ddopai.dataloaders.base import BaseDataLoader
from ddopai.loss_functions import pinball_loss, quantile_loss
from ddopai.envs.pricing.base import BasePricingEnv

import gymnasium as gym

import numpy as np
import time

In [ ]:
# | export
class DynamicPricingEnv(BasePricingEnv):
    """
    Class implementing the dynamic pricing and learning problem, working for the single- and multi-item case.
    If alpha and beta are scalars and they are multiple SKUs, then the same parameters are used for all SKUs.
    If alpha and beta are arrays, then they should have the same length as the number of SKUs.
    Num_SKUs can be set as parameter or inferrred from the DataLoader.
    """
    def __init__(self,
        alpha: Union[np.ndarray, Parameter, int, float] = 1.0, # market size per SKUs
        beta: Union[np.ndarray, Parameter, int, float] = 0.5, # price elasticity per SKUs
        p_bound_low: Union[np.ndarray, Parameter, int, float] = 0.0, # lower price bound per SKUs
        p_bound_high: Union[np.ndarray, Parameter, int, float] = 1.0, # upper price bound per SKUs
        dataloader: BaseDataLoader = None, # dataloader TODO: replace with pricing dataloader
        num_SKUs: Union[np.ndarray, Parameter, int, float] = None, # number of SKUs
        gamma: float = 1, # discount factor
        horizon_train: int | str = "use_all_data", # if "use_all_data" then horizon is inferred from the DataLoader
        postprocessors: list[object] | None = None, # default is empty list 
        mode: str = "online", # TODO: add online to relevant modes
        return_truncation: str = True # TODO:Why is this a string?
        ) -> None:

        self.print=False
        
        num_SKUs = dataloader.num_units if num_SKUs is None else num_SKUs
        
        if not isinstance(num_SKUs, int):
            raise ValueError("num_SKUs should be an integer.")
        
        self.set_param("num_SKUs", num_SKUs, shape=(1,), new=True)
        
        self.set_param("p_bound_low", p_bound_low, shape=(num_SKUs,), new=True)
        self.set_param("p_bound_high", p_bound_high, shape=(num_SKUs,), new=True)
        
        self.set_observation_space(dataloader.X_shape)
        self.set_action_space(dataloader.Y_shape, low = self.p_bound_low, high = self.p_bound_high)
        
        mdp_info = MDPInfo(self.observation_space, self.action_space, gamma=gamma, horizon=horizon_train)
        
        super().__init__(mdp_info=mdp_info,
                         postprocessors=postprocessors,
                         mode=mode, return_truncation=return_truncation,
                         alpha=alpha,
                         beta=beta,
                         dataloader=dataloader,
                         horizon_train=horizon_train)
        
    def step_(self,
              action: np.ndarray # prices)
                ) -> Tuple[np.ndarray, float, bool, bool, dict]:
        return observation, reward, terminated, truncated, info